# Reading Our Staged Datasets With Autoloaders


## DimArtist Ingestion/Transformations with Autoloaders

In [0]:
df_artist  = spark.readStream.format("cloudFiles").option("cloudFiles.format","parquet").option("cloudFiles.schemaLocation", "abfss://silver@jessdatalake.dfs.core.windows.net/DimArtist/checkpoint_location").option("schemaEvolutionMode", "addNewColumns").load("abfss://bronze@jessdatalake.dfs.core.windows.net/DimArtist")

In [0]:
df_artist.display()

In [0]:
df_artist = df_artist.drop("_rescued_data")\
    .dropDuplicates(["artist_id"])
df_artist.display()


In [0]:
df_artist.writeStream.format("delta").outputMode("append").option("checkpointLocation", "abfss://silver@jessdatalake.dfs.core.windows.net/DimArtist/checkpoint_location").option("path", "abfss://silver@jessdatalake.dfs.core.windows.net/DimArtist/dimartist").trigger(once=True).toTable("musicstreaming_project.silver.dimartist")

## DimTrack Ingestion/Transformations with Autoloaders

In [0]:
df_track = spark.readStream.format("cloudFiles").option("cloudFiles.format", "parquet").option("cloudFiles.SchemaLocation", "abfss://silver@jessdatalake.dfs.core.windows.net/DimTrack/checkpoint_location").option("cloudFiles.schemaEvolutionMode", "addNewColumns").load("abfss://bronze@jessdatalake.dfs.core.windows.net/DimTrack")
df_track.display()


In [0]:
df_track = df_track.drop("_rescued_data")


In [0]:
df_track = df_track.withColumn("duration_flag", when(col("duration_sec")< 150, "short")\
    .when(col("duration_sec")>=150, "medium")\
    .otherwise("long"))

df_track.display()

In [0]:
df_track = df_track.withColumn("track_name", regexp_replace(col("track_name"), "-", " "))
df_track.display()

In [0]:
df_track.writeStream.format("delta").outputMode("append").option("checkpointLocation", "abfss://silver@jessdatalake.dfs.core.windows.net/DimTrack/checkpoint_location").option("path", "abfss://silver@jessdatalake.dfs.core.windows.net/DimTrack/dimtrack").trigger(once=True).toTable("musicstreaming_project.silver.dimtrack")

## DimUser Ingestion/Transformations with Autoloaders

In [0]:
df = spark.readStream.format("cloudFiles").option("cloudFiles.format","parquet").option("cloudFiles.schemaLocation", "abfss://silver@jessdatalake.dfs.core.windows.net/DimUser/checkpoint_location").option("schemaEvolutionMode", "addNewColumns").load("abfss://bronze@jessdatalake.dfs.core.windows.net/DimUser")

In [0]:
df.display()

In [0]:
from pyspark.sql.functions import *

In [0]:
df = df.withColumn("subscription_type", regexp_replace("subscription_type", "Premium", "Premium Plan"))\
    .withColumn("subscription_type", regexp_replace("subscription_type", "Free", "Free Plan"))\
    .withColumn("subscription_type", regexp_replace("subscription_type", "Family", "Family Plan"))


In [0]:
df = df.drop("_rescued_data")
df.display()


In [0]:
df = df.dropDuplicates(["user_id"])
df.display()

In [0]:
df = df.writeStream.format("delta").outputMode("append").option("checkpointLocation", "abfss://silver@jessdatalake.dfs.core.windows.net/DimUser/checkpoint_location").option("path", "abfss://silver@jessdatalake.dfs.core.windows.net/DimUser/dimuser").trigger(once=True).toTable("musicstreaming_project.silver.dimuser")

## DimDate Ingestion/Transformations with Autoloaders

In [0]:
df_date = spark.readStream.format("cloudFiles").option("cloudFiles.format", "parquet").option("cloudFiles.SchemaLocation", "abfss://silver@jessdatalake.dfs.core.windows.net/DimDate/checkpoint_location").option("cloudFiles.schemaEvolutionMode", "addNewColumns").load("abfss://bronze@jessdatalake.dfs.core.windows.net/DimDate")
df_date.display()

In [0]:
df_date = df_date.drop("_rescued_data")
df_date.display()
df_date.writeStream.format("delta").outputMode("append").option("checkpointLocation", "abfss://silver@jessdatalake.dfs.core.windows.net/DimDate/checkpoint_location").option("path", "abfss://silver@jessdatalake.dfs.core.windows.net/DimDate/dimdate").trigger(once=True).toTable("musicstreaming_project.silver.dimdate")

## FactStreams Ingestion/Transformations with Autoloaders

In [0]:
df_fact = spark.readStream.format("cloudFiles").option("cloudFiles.format", "parquet").option("cloudFiles.SchemaLocation", "abfss://silver@jessdatalake.dfs.core.windows.net/FactStream/checkpoint_location").option("cloudFiles.schemaEvolutionMode", "addNewColumns").load("abfss://bronze@jessdatalake.dfs.core.windows.net/FactStream")

In [0]:
df_fact.display()

In [0]:
df_fact = df_fact.drop("_rescued_data")
df_fact.display()
df_fact.writeStream.format("delta").outputMode("append").option("checkpointLocation", "abfss://silver@jessdatalake.dfs.core.windows.net/FactStream/checkpoint_location").option("path", "abfss://silver@jessdatalake.dfs.core.windows.net/FactStream/factstream").trigger(once=True).toTable("musicstreaming_project.silver.factstream")

In [0]:
%sql
SELECT * FROM musicstreaming_project.gold.dimuser